**Neste steg i Data Science-prosessen er å velge en modell for prediksjon**

Først splitter jeg opp den ferdig behandlede dataen min i en trenings, validerings og testdel.

In [107]:
# Importerer nødvendige pakker

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
import plotly.express as px

In [98]:
# Leser inn den ferdig behandlede dataen, og splitter opp
# Bruker IKKE train_test_split for å unngå tidslekasjer

data = pd.read_csv("model_ready.csv")
data_encoded = pd.get_dummies(data, columns=["station"], drop_first=True)
data_encoded = data_encoded.dropna() # Fjerner NaN rader

target = "fb_next_h" # target verdien
X_cols = [
    'free_bikes',     # current state
    'free_spots',     # current capacity info
    'trips_in',       # activity indicators  
    'trips_out',
    'temperature',    # weather features
    'precipitation',
    'wind_speed',
    'hour',          # temporal features
    'weekday',
    'fb_prev_h'      # lag feature
]
station_cols = [col for col in data_encoded.columns if col.startswith('station_')]

X = data_encoded[X_cols + station_cols]
y = data_encoded[target]


n = len(data_encoded)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_data = data_encoded.iloc[:train_end]
val_data = data_encoded.iloc[train_end:val_end]
test_data = data_encoded.iloc[val_end:]

train_X, train_y = train_data[X_cols + station_cols], train_data[target]
val_X, val_y = val_data[X_cols + station_cols], val_data[target]
test_X, test_y = test_data[X_cols + station_cols], test_data[target]

Etter dataen er splittet opp må jeg velge en god modell. Etter som target-verdien er kontinuerlig går jeg for en regresjonsmodell.

In [99]:
# trener baseline modeller

baseline_models = {"Dummy_mean" : DummyRegressor(strategy="mean"),
          "Dummy_median" : DummyRegressor(strategy="median"),}

for name, model in baseline_models.items():
    model.fit(train_X, train_y)
    y_val_pred = model.predict(val_X)
    rmse_model = np.sqrt(mean_squared_error(val_y, y_val_pred))
    print(f"{name} : med en RMSE på {rmse_model}")

Dummy_mean : med en RMSE på 4.920988386106658
Dummy_median : med en RMSE på 3.392423995861449


In [100]:
# trener andre modeller

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

ml_models = {
    "Linear": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "KNN": KNeighborsRegressor(n_neighbors=5), # tester n=5
    "SVR": SVR(kernel='rbf'),
    "Lasso": Lasso(alpha=0.1),
    "Ridge": Ridge(alpha=1.0)
}

results = {} # lager en dic for rmse resultatene til modellene

for name, model in ml_models.items():
    print(f"Trener {name}...")
    model.fit(train_X, train_y)
    y_val_pred = model.predict(val_X)
    rmse = np.sqrt(mean_squared_error(val_y, y_val_pred))
    results[name] = rmse
    print(f"{name}: RMSE = {rmse:.3f}")

# finner beste modellen
best_model = min(results, key=results.get)
best_rmse = results[best_model]
print(f"Beste modellen er {best_model} med en RMSE = {best_rmse:.3f}")

Trener Linear...
Linear: RMSE = 1.277
Trener RandomForest...
RandomForest: RMSE = 1.202
Trener GradientBoosting...
GradientBoosting: RMSE = 1.199
Trener KNN...
KNN: RMSE = 1.173
Trener SVR...
SVR: RMSE = 1.277
Trener Lasso...
Lasso: RMSE = 1.222
Trener Ridge...
Ridge: RMSE = 1.277
Beste modellen er KNN med en RMSE = 1.173


In [101]:
# finner beste modellen
best_model = min(results, key=results.get)
best_rmse = results[best_model]
print(f"Beste modellen er {best_model} med en RMSE = {best_rmse:.3f}")

Beste modellen er KNN med en RMSE = 1.173


Men jeg vil også sjekke balansen til modellene. Altså overfitting og underfitting. Dette sjekker generalisering til modellen, altså hvor god den vil være på nye eller mer data eventuelt. Da sammenligner jeg resultatene av trenings og valideringsdata.

In [102]:
# først trener jeg og sjekker mot samme treningsdata
train_results = {}

for name, model in ml_models.items():
    print(f"Trener {name}...")
    model.fit(train_X, train_y)
    y_train_pred = model.predict(train_X)
    rmse = np.sqrt(mean_squared_error(train_y, y_train_pred))
    train_results[name] = rmse
    print(f"{name}: RMSE = {rmse:.3f}")

print()

# sammeligner de to settene av resultater

for name in results.keys():
    train_rmse = train_results[name]
    val_rmse = results[name]
    diff = val_rmse - train_rmse

    print(f"{name}:(Diff: {diff:.3f})")

Trener Linear...
Linear: RMSE = 1.100
Trener RandomForest...
RandomForest: RMSE = 0.421
Trener GradientBoosting...
GradientBoosting: RMSE = 1.043
Trener KNN...
KNN: RMSE = 1.079
Trener SVR...
SVR: RMSE = 1.090
Trener Lasso...
Lasso: RMSE = 1.112
Trener Ridge...
Ridge: RMSE = 1.100

Linear:(Diff: 0.178)
RandomForest:(Diff: 0.781)
GradientBoosting:(Diff: 0.156)
KNN:(Diff: 0.095)
SVR:(Diff: 0.187)
Lasso:(Diff: 0.110)
Ridge:(Diff: 0.178)


Ut ifra resultatene er KNN-modellen den klare vinneren med både best ytelse og balanse.
Det neste jeg da kan gjøre er å sjekke ulike n-verdier av KNN:

In [103]:
# Tester ulike n-verdier for KNN

n_values = [1, 3, 5, 7, 10, 15, 20, 25, 30]
knn_results = {}

for n in n_values:
    knn_model = KNeighborsRegressor(n_neighbors=n)
    knn_model.fit(train_X, train_y)
    
    # Evaluer på både train og validation
    y_train_pred = knn_model.predict(train_X)
    y_val_pred = knn_model.predict(val_X)
    
    train_rmse = np.sqrt(mean_squared_error(train_y, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(val_y, y_val_pred))
    
    knn_results[n] = {'train': train_rmse, 'val': val_rmse, 'diff': val_rmse - train_rmse}
    
    print(f"n={n}: Train={train_rmse:.3f}, Val={val_rmse:.3f}, Diff={val_rmse - train_rmse:.3f}")


n=1: Train=0.000, Val=1.455, Diff=1.455
n=3: Train=0.968, Val=1.226, Diff=0.259
n=5: Train=1.079, Val=1.173, Diff=0.095
n=7: Train=1.143, Val=1.148, Diff=0.005
n=10: Train=1.198, Val=1.129, Diff=-0.069
n=15: Train=1.254, Val=1.114, Diff=-0.140
n=20: Train=1.292, Val=1.110, Diff=-0.182
n=25: Train=1.319, Val=1.112, Diff=-0.207
n=30: Train=1.343, Val=1.113, Diff=-0.230


**Tolkning**

n=20 gir beste rmse, med en liten underfitting på -0.182 i differanse, som er godkjent i generalisering-forstand.
En veldig lav n gir ekstrem overfitting, mens n=7 gir nærmest perfekt balanse.
På n=10 og oppover begynner underfitting, modellen blir for enkel.

Velger KNN modellen med n=20 som min modell, og tester den til slutt mot testdataen

In [104]:
final_model = KNeighborsRegressor(n_neighbors=20)
final_model.fit(train_X, train_y)

y_final_pred = final_model.predict(test_X)

print(np.sqrt(mean_squared_error(test_y, y_final_pred)))

1.8845589773854263


**Visualiseringer**

In [109]:

# Modellene sin ytelse
models = list(results.keys())
scores = list(results.values())

fig1 = px.bar(x=models, y=scores, 
              title="Modell Performance (Validation RMSE)",
              labels={'x': 'Modeller', 'y': 'RMSE'})
fig1.show()

# KNN hyperparameter tuning
n_vals = [1, 3, 5, 7, 10, 15, 20, 25, 30]
val_scores = [knn_results[n]['val'] for n in n_vals]

fig2 = px.line(x=n_vals, y=val_scores, 
               title="KNN: n_neighbors vs RMSE",
               labels={'x': 'n_neighbors', 'y': 'Validation RMSE'},
               markers=True)
fig2.show()